In [1]:
import os
from dataset_utils import *
from torch.utils.data import DataLoader
import torch.nn.functional as F

In [2]:
import os
data_path = "/home/linhang/workbench/Earthquake_data/"
os.listdir(data_path)

['Japan', 'California']

In [3]:
area = "Japan"

In [4]:
data_area_path = data_path + area + "/"
gnss_data = pd.read_csv(data_area_path + "gnss_data.csv", index_col=0, parse_dates=True, low_memory=False).map(parse_str_list)


In [5]:
gnss_geo_matrix = pd.read_csv(data_area_path+"gnss_geo_matrix.csv", index_col=0)

In [130]:
idx = 0 
window_size = 300

In [131]:
gnss_data_history = gnss_data.iloc[idx:idx + window_size].values

In [132]:
gnss_data_history = np.array([convert_to_fixed_length_array(row) for row in gnss_data_history.T])

In [135]:
missing_mask = np.isnan(gnss_data_history).all(axis=2)

In [139]:
def max_consecutive_trues(arr):
    """
    Efficiently compute the maximum number of consecutive True values along axis 1 for each row.

    Parameters:
    - arr: Boolean numpy array of shape (num_stations, window_size)

    Returns:
    - Numpy array of shape (num_stations,) containing max consecutive True counts per station
    """
    # Convert boolean array to int
    arr = arr.astype(int)

    # Pad the array with zeros at both ends
    padded = np.pad(arr, ((0, 0), (1, 1)), 'constant', constant_values=0)

    # Find where the differences occur
    diff = np.diff(padded, axis=1)

    # Start indices of sequences of 1s (True values)
    run_starts = np.where(diff == 1)

    # End indices of sequences of 1s (True values)
    run_ends = np.where(diff == -1)

    # Compute lengths of the runs
    run_lengths = run_ends[1] - run_starts[1]

    # Initialize max_lengths array
    max_lengths = np.zeros(arr.shape[0], dtype=int)

    # Use np.maximum.at to efficiently compute the maximum run length per station
    np.maximum.at(max_lengths, run_starts[0], run_lengths)

    return max_lengths

In [140]:
max_missing_lengths = max_consecutive_trues(missing_mask)

In [141]:
max_missing_lengths

array([300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300,
       300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300,
       300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300,
       300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300,
       300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300,
       300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300,
       300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300,
       300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300,
       300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300,
       300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300,
       300, 300,   4, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300,
       300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300,
       300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300,
       300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 30

In [142]:
stations_to_keep = np.where(max_missing_lengths <= 30)[0]

In [143]:
gnss_data_history = gnss_data_history[stations_to_keep]

In [144]:
gnss_data_history

array([[[        nan,         nan,         nan,         nan],
        [ 0.706415  ,  0.741343  ,  0.222997  ,  1.04801682],
        [ 0.704001  ,  0.740332  ,  0.215487  ,  1.04409938],
        ...,
        [        nan,         nan,         nan,         nan],
        [        nan,         nan,         nan,         nan],
        [        nan,         nan,         nan,         nan]],

       [[ 0.934276  , -0.8422    ,  0.266355  ,  1.28573616],
        [ 0.934069  , -0.844145  ,  0.265954  ,  1.28677784],
        [ 0.933887  , -0.845971  ,  0.265005  ,  1.28764883],
        ...,
        [ 0.926541  , -0.840757  ,  0.247401  ,  1.27536576],
        [ 0.925689  , -0.841398  ,  0.246978  ,  1.27508778],
        [ 0.926861  , -0.838625  ,  0.248443  ,  1.27439677]]])

In [154]:
# 如果 gnss_geo_matrix 是 DataFrame
sample_gnss_geo_mask = gnss_geo_matrix[np.ix_(stations_to_keep, stations_to_keep)]

InvalidIndexError: (array([[132],
       [348]]), array([[132, 348]]))

In [148]:
gnss_data_history = fill_nan_with_interpolation(gnss_data_history)

In [138]:
gnss_geo_matrix

array([[  0.        , 345.75040042],
       [345.75040042,   0.        ]])